# 07 — Mamba / SSM: Temporal Features for Velocity & Timing (L_V, L_T)

Train a discretized state-space model on per-card transaction sequences.
Extract scalar behavioral features and test via LR test.

**Primary label targets:** `L_V` (velocity burst), `L_T` (temporal anomaly)  
**Nested test:** M_{+GNN} ⊂ M_{+GNN+Mamba}

CPU fallback (DiscretizedSSM) is used here.  
For GPU: `pip install mamba-ssm causal-conv1d` and swap the model in `src/models/ssm.py`.

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
sys.path.insert(0, '..')

from src.labels import LABEL_COLS
from src.features import build_features, label_matrix
from src.models.ssm import MambaExtractor
from src.models.glm import BinaryRelevanceGLM
from src.evaluation import lr_test

In [ ]:
train = pd.read_parquet('../data/processed/train_labeled.parquet')
test  = pd.read_parquet('../data/processed/test_labeled.parquet')

X_train = build_features(train)
X_test  = build_features(test)
y_train = label_matrix(train)
y_test  = label_matrix(test)

In [ ]:
# Train SSM — this may take a few minutes on CPU
mamba = MambaExtractor(d_state=32, epochs=20, batch_size=64, max_seq_len=256)
mamba.fit(train)
print('Training complete.')

In [ ]:
# Extract features
ssm_feats_train = mamba.extract_features(train)
ssm_feats_test  = mamba.extract_features(test)
print(ssm_feats_train.describe())

In [ ]:
# Load GNN features for the combined X
gnn_train = pd.read_parquet('../data/processed/gnn_feats_train.parquet')
gnn_test  = pd.read_parquet('../data/processed/gnn_feats_test.parquet')

X_plus_gnn_train = pd.concat([X_train, gnn_train], axis=1)
X_plus_gnn_test  = pd.concat([X_test,  gnn_test],  axis=1)

In [ ]:
# LR test: does Mamba add over M_{+GNN}?
br = BinaryRelevanceGLM()
for label in LABEL_COLS:
    result = br.admit_extension(X_plus_gnn_train, ssm_feats_train, y_train, label)
    sym = '✓ ADMITTED' if result['admitted'] else '✗ dropped'
    print(f'{label}: G²={result["G2"]:.2f}  p={result["p_value"]:.4f}  {sym}')

In [ ]:
# Save Mamba features
ssm_feats_train.to_parquet('../data/processed/ssm_feats_train.parquet')
ssm_feats_test.to_parquet('../data/processed/ssm_feats_test.parquet')
print('Saved Mamba features.')